# Ingest — Copilot Studio consumption, via API

Reads Copilot Credit consumption from the **Power Platform licensing API** into
`studio_consumption_daily`, `studio_capacity` and `studio_environment`.

This is the automated alternative to [`Ingest_Studio`](Ingest_Studio.ipynb), which reads the
manually downloaded Power Platform admin center CSVs. Prefer this notebook where you can: it
needs no one to visit the portal, and it solves the problem that makes the CSV path awkward.

## Why this is worth switching to

`StudioPerAgent.csv` has **no date column** — it is a month-to-date aggregate, so `Ingest_Studio`
has to stamp each load with a `snapshot_month` and can only ever produce cumulative period
totals. The API takes `fromDate` and `toDate`, so asking for one day at a time returns a genuine
**daily** per-agent history. That is the grain the forecast page wants and the CSV cannot supply.

## What it does not give you

There is **no per-user consumption route**. `StudioPerUser.csv` remains a manual download, and
`studio_user` is still populated by `Ingest_Studio`. The two notebooks write disjoint tables and
can be scheduled alongside each other.

## Endpoints

| | |
|---|---|
| `GET /licensing/entitlements/MCSMessages` | tenant capacity: entitled, allocated, consumed, available |
| `GET /licensing/entitlements/MCSMessages/resources` | per-resource consumption for a date range |
| `GET /environmentmanagement/environments` | environment names, which the consumption rows omit |

`MCSMessages` is the entitlement ID for Copilot Credits. It is still the identifier after the
rename to Copilot Credits; do not substitute a friendlier-looking string.

## Source conventions to check in your tenant

- **The response model is only partly documented.** The REST reference types `metadata` as a bare
  object — "additional metadata such as Feature, ProductName and nonBillableConsumed". The richer
  dimensions and the `includeFields` parameter are not in the public reference at all. This
  notebook therefore reads every metadata key case-insensitively, promotes the ones the model
  uses, and keeps the whole object as JSON in `metadata_json` so nothing is lost when Microsoft
  adds a field. **Check the fields your report depends on after any platform update.**
- **Detail depends on the harness.** Standard harness agents can report feature, tool, model,
  channel and knowledge source. **GitHub Copilot harness agents currently report the feature as
  `Process Agent` and return no tool, model or knowledge-source values.** Blank is the source
  telemetry, not a collection failure — so this notebook keeps real nulls rather than inventing
  defaults, and derives `harness_hint` from that convention rather than asserting it as fact.
- **Multiple rows per agent per day are normal.** The API splits on channel, feature, model and
  environment. Aggregate at read time; do not assume one row per agent per day.
- **`lastRefreshedDate` is the honest watermark.** The entitlement response also carries the
  latest completed usage date. Recent days can still be restating, which is why this notebook
  re-collects a trailing window rather than only appending yesterday.

Authentication is explicit: an Entra application credential is read from Key Vault and exchanged
for an `https://api.powerplatform.com` token. Fabric's `getToken` audiences do **not** include the
Power Platform API; do not assume a notebook managed identity works. The application needs the
Power Platform API permissions described in the
[advanced setup guide](../../docs/ADVANCED-SETUP.md#azure-ingestion-in-fabric), and the execution
identity needs Key Vault secret-read and Lakehouse write access.

All collection and validation completes before anything is written. Refresh the semantic model
only after the whole notebook succeeds; the Delta writes are not one transaction.

## Configure

In [ ]:
TENANT_ID = "00000000-0000-0000-0000-000000000000"
CLIENT_ID = "00000000-0000-0000-0000-000000000000"
KEY_VAULT_URL = "https://<your-vault>.vault.azure.net/"
CLIENT_SECRET_NAME = "consumption-central-powerplatform-client-secret"

# Days of history to collect, ending yesterday (UTC). The first run should use a
# larger number to backfill; later runs only need to cover the restatement
# window. 180 is the practical ceiling for the source.
DAYS = 180

# Days at the end of the range that are re-collected and overwritten every run,
# because recent consumption restates. Rows inside this window are replaced,
# not merged, so a row the source has withdrawn disappears here too.
RESTATE_DAYS = 7

# Rows per page. The API fans out per environment, so this is a ceiling rather
# than an exact page size.
PAGE_SIZE = 5000

# Undocumented but supported in practice: asks for the richer dimensions.
# Set to None to request only the documented response.
INCLUDE_FIELDS = "users,tags,asOfDate"

TBL_CONSUMPTION = "studio_consumption_daily"
TBL_CAPACITY = "studio_capacity"
TBL_ENVIRONMENT = "studio_environment"

In [ ]:
import json
import math
import time
import requests
from datetime import date, datetime, timedelta, timezone
from email.utils import parsedate_to_datetime
from urllib.parse import urlsplit, urlencode
from uuid import UUID

PPAPI = "https://api.powerplatform.com"
PPAPI_HOST = "api.powerplatform.com"
API_VERSION = "2024-10-01"
ENTITLEMENT = "MCSMessages"

_token = None
_token_until = 0


def pp_headers():
    global _token, _token_until
    if _token is None or time.monotonic() >= _token_until:
        secret = notebookutils.credentials.getSecret(KEY_VAULT_URL, CLIENT_SECRET_NAME)
        r = requests.post(
            f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token",
            data={"grant_type": "client_credentials", "client_id": CLIENT_ID,
                  "client_secret": secret, "scope": f"{PPAPI}/.default"},
            timeout=90, allow_redirects=False)
        r.raise_for_status()
        payload = r.json()
        _token = payload["access_token"]
        _token_until = time.monotonic() + max(0, int(payload["expires_in"]) - 300)
    return {"Authorization": f"Bearer {_token}", "Accept": "application/json"}


def retry_delay(headers, attempt):
    """Honour the server's own backoff, falling back to exponential."""
    delays = [float(2 ** attempt)]
    for name, value in headers.items():
        key = name.lower()
        if key == "retry-after" or (key.startswith("x-ms-ratelimit-")
                                    and key.endswith("retry-after")):
            try:
                delay = float(value)
            except ValueError:
                if key != "retry-after":
                    raise ValueError("Invalid licensing retry header") from None
                retry_at = parsedate_to_datetime(value)
                if retry_at.tzinfo is None:
                    retry_at = retry_at.replace(tzinfo=timezone.utc)
                delay = (retry_at - datetime.now(timezone.utc)).total_seconds()
            if not math.isfinite(delay):
                raise ValueError("Invalid retry delay")
            delays.append(max(0, delay))
    delay = max(delays)
    if delay > 300:
        raise RuntimeError("Server retry delay exceeds 300 seconds; defer the job")
    return delay


def pp_get(path, **params):
    """GET against the Power Platform API, with retry and a credential guard.

    The host is pinned: a continuation token is opaque and must never cause a
    bearer token to be sent somewhere other than the licensing service.
    """
    params = {k: v for k, v in params.items() if v is not None}
    params["api-version"] = API_VERSION
    url = f"{PPAPI}{path}?{urlencode(params)}"

    parsed = urlsplit(url)
    if (parsed.scheme != "https" or parsed.netloc.lower() != PPAPI_HOST
            or parsed.fragment):
        raise ValueError("Refusing to send credentials off the Power Platform API")

    for attempt in range(5):
        r = requests.get(url, headers=pp_headers(), timeout=180, allow_redirects=False)
        if r.status_code in (429, 502, 503, 504) and attempt < 4:
            delay = retry_delay(r.headers, attempt)
            print(f"  HTTP {r.status_code}; retrying in {delay:g}s")
            r.close()
            time.sleep(delay)
            continue
        if r.status_code == 403:
            raise PermissionError(
                "403 from the licensing API. The application needs Power Platform API "
                "licensing permissions and admin consent; Dataverse or Graph consent "
                "does not grant this.")
        r.raise_for_status()
        if r.status_code != 200:
            raise RuntimeError(f"Unexpected licensing status {r.status_code}")
        return r.json()


def pp_pages(path, **params):
    """Follow continuation tokens until the service stops issuing them."""
    token = None
    seen = 0
    for page in range(1, 501):
        payload = pp_get(path, continuationtoken=token, **params)
        rows = payload.get("value") or []
        seen += len(rows)
        yield rows
        token = payload.get("continuationtoken") or payload.get("continuationToken")
        if not token:
            return
    raise RuntimeError(f"Pagination did not terminate after {seen:,} rows")


def meta_get(metadata, *aliases):
    """Case- and separator-insensitive lookup into the loose metadata object.

    The reference types this as a bare object, so key casing is not guaranteed
    stable. Normalising costs nothing and removes a whole class of silent
    breakage on a platform update.
    """
    if not isinstance(metadata, dict):
        return None
    lookup = {}
    for key, value in metadata.items():
        lookup.setdefault("".join(ch for ch in str(key).lower() if ch.isalnum()), value)
    for alias in aliases:
        norm = "".join(ch for ch in alias.lower() if ch.isalnum())
        if norm in lookup:
            value = lookup[norm]
            if value is None:
                return None
            if isinstance(value, (list, tuple)):
                value = ", ".join(str(v) for v in value if v is not None)
            text = str(value).strip()
            return text or None
    return None


def as_float(value):
    try:
        out = float(value)
    except (TypeError, ValueError):
        return None
    return out if math.isfinite(out) else None


for identifier in (TENANT_ID, CLIENT_ID):
    if UUID(identifier).int == 0:
        raise ValueError("Configure tenant and application client IDs before running")
if not isinstance(DAYS, int) or isinstance(DAYS, bool) or not 1 <= DAYS <= 180:
    raise ValueError("DAYS must be an integer from 1 to 180")
if not isinstance(RESTATE_DAYS, int) or isinstance(RESTATE_DAYS, bool) or not 1 <= RESTATE_DAYS <= DAYS:
    raise ValueError("RESTATE_DAYS must be an integer from 1 to DAYS")

END = datetime.now(timezone.utc).date() - timedelta(days=1)
START = END - timedelta(days=DAYS - 1)
print(f"collecting {START} to {END} ({DAYS} days), restating the last {RESTATE_DAYS}")

## 1. Tenant capacity

One row per run. `lastRefreshedDate` (or whichever key the service uses for it) is the watermark
telling you how current the detailed consumption below actually is — worth reading before
concluding that yesterday looks quiet.

In [ ]:
ent = pp_get(f"/licensing/entitlements/{ENTITLEMENT}")

capacity_row = {
    "entitlement_id": ENTITLEMENT,
    "entitled": as_float(meta_get(ent, "entitled", "entitledQuantity", "totalEntitled")),
    "allocated": as_float(meta_get(ent, "allocated", "allocatedQuantity")),
    "consumed": as_float(meta_get(ent, "consumed", "consumedQuantity")),
    "available": as_float(meta_get(ent, "available", "availableQuantity")),
    "payg_consumed": as_float(meta_get(ent, "payAsYouGoConsumed", "paygConsumed",
                                       "payAsYouGoConsumedQuantity")),
    "status": meta_get(ent, "status"),
    "unit": meta_get(ent, "unit"),
    "last_usage_date": meta_get(ent, "lastCompletedUsageDate", "latestCompletedUsageDate",
                                "lastRefreshedDate", "asOfDate"),
    "payload_json": json.dumps(ent, sort_keys=True)[:1048576],
}

if capacity_row["consumed"] is None and capacity_row["entitled"] is None:
    raise RuntimeError(
        "The entitlement response carried neither consumed nor entitled. The shape has "
        f"changed; inspect it before trusting this load. Keys: {sorted(ent)}")

print(json.dumps({k: v for k, v in capacity_row.items() if k != "payload_json"},
                 indent=2, default=str))
watermark = capacity_row["last_usage_date"]
if watermark:
    print(f"\nlatest completed usage date reported by the service: {watermark}")

## 2. Environments

The consumption rows carry `environmentId` but no name, so this resolves them. It is a small,
cheap call and the join is far more useful than a grid full of GUIDs.

In [ ]:
env_rows = []
for page in pp_pages("/environmentmanagement/environments"):
    for e in page:
        props = e.get("properties") if isinstance(e.get("properties"), dict) else {}
        env_id = (meta_get(e, "id", "name", "environmentId")
                  or meta_get(props, "environmentId"))
        if not env_id:
            continue
        # The id can arrive as a full ARM-style path; the trailing segment is the GUID.
        env_id = env_id.rstrip("/").split("/")[-1]
        env_rows.append({
            "environment_id": env_id.lower(),
            "environment_name": (meta_get(props, "displayName", "name")
                                 or meta_get(e, "displayName") or env_id),
            "environment_type": meta_get(props, "environmentSku", "environmentType", "sku"),
            "region": meta_get(props, "azureRegion", "location") or meta_get(e, "location"),
        })

env_lookup = {r["environment_id"]: r["environment_name"] for r in env_rows}
print(f"{len(env_rows):,} environments")
if not env_rows:
    print("  ! none returned - consumption will still load, with GUIDs in place of names")

## 3. Daily per-agent consumption

One request per day, because a multi-day range collapses the dates and the whole point of using
the API is to get the daily grain the CSV export cannot provide.

`harness_hint` is **inferred**, not reported. GitHub Copilot harness agents currently surface
`Process Agent` as the feature with no tool, model or knowledge source; standard harness agents
surface real feature detail. That is a documented convention, not a contract, so the column is
named as a hint and the raw feature is kept beside it. For an authoritative split, join the
**Harness** column from Power Platform inventory on agent id.

In [ ]:
def collect_day(day):
    out = []
    for page in pp_pages(f"/licensing/entitlements/{ENTITLEMENT}/resources",
                         fromDate=day.isoformat(), toDate=day.isoformat(),
                         includeFields=INCLUDE_FIELDS, pageSize=PAGE_SIZE):
        for row in page:
            md = row.get("metadata") if isinstance(row.get("metadata"), dict) else {}
            env_id = (meta_get(row, "environmentId") or "").lower()
            feature = meta_get(md, "feature", "featureName")
            tool = meta_get(md, "tool", "toolUsed", "tools")
            model = meta_get(md, "model", "llmModel", "modelName")
            knowledge = meta_get(md, "knowledgeSource", "knowledgeSources", "knowledge")

            # Inference, deliberately conservative: only the documented signature.
            if feature and feature.strip().lower() == "process agent":
                harness_hint = "GitHub Copilot (inferred)"
            elif feature or tool or model or knowledge:
                harness_hint = "Standard (inferred)"
            else:
                harness_hint = None

            out.append({
                "usage_date": day,
                "resource_id": meta_get(row, "resourceId"),
                "resource_name": meta_get(md, "resourceName", "displayName", "name"),
                "environment_id": env_id or None,
                "environment_name": env_lookup.get(env_id),
                "product_name": meta_get(md, "productName", "product"),
                "feature": feature,
                "channel": meta_get(md, "channel", "channelName"),
                "tool_used": tool,
                "llm_model": model,
                "knowledge_sources": knowledge,
                "harness_hint": harness_hint,
                "billed_credit": as_float(row.get("consumed")),
                "non_billed_credit": as_float(
                    meta_get(md, "nonBillableConsumed", "nonBilledConsumed",
                             "nonBillableConsumption")),
                "reported_users": as_float(meta_get(md, "users", "userCount",
                                                    "reportedUsers")),
                "unit": meta_get(row, "unit"),
                "as_of_date": meta_get(md, "asOfDate") or meta_get(row, "lastRefreshedDate"),
                "metadata_json": json.dumps(md, sort_keys=True)[:1048576] if md else None,
            })
    return out


rows = []
empty_days = 0
day = START
while day <= END:
    got = collect_day(day)
    rows.extend(got)
    if not got:
        empty_days += 1
    if day.day == 1 or day == END:
        print(f"  {day}  running total {len(rows):,} rows")
    day += timedelta(days=1)

print(f"\n{len(rows):,} consumption rows over {DAYS} days "
      f"({empty_days} day(s) returned nothing)")

if rows and all(r["resource_id"] is None for r in rows):
    raise RuntimeError("Every row is missing resourceId; the response shape has changed")
if empty_days == DAYS:
    print("  ! nothing returned for any day. Either the tenant consumed no credits in "
          "this window, or the application lacks licensing read permission.")

## 4. Write

Days inside the restatement window are **replaced**, so a row the source has withdrawn disappears
here too. Older days are merged, which keeps history that has since fallen out of the API's
retention. Nothing is written until every collection step above has succeeded.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               DateType, TimestampType)
from delta.tables import DeltaTable

CONSUMPTION_SCHEMA = StructType([
    StructField("usage_date", DateType()),
    StructField("resource_id", StringType()),
    StructField("resource_name", StringType()),
    StructField("environment_id", StringType()),
    StructField("environment_name", StringType()),
    StructField("product_name", StringType()),
    StructField("feature", StringType()),
    StructField("channel", StringType()),
    StructField("tool_used", StringType()),
    StructField("llm_model", StringType()),
    StructField("knowledge_sources", StringType()),
    StructField("harness_hint", StringType()),
    StructField("billed_credit", DoubleType()),
    StructField("non_billed_credit", DoubleType()),
    StructField("reported_users", DoubleType()),
    StructField("unit", StringType()),
    StructField("as_of_date", StringType()),
    StructField("metadata_json", StringType()),
])

# A resource appears once per feature, channel, model and environment per day.
# Every one of those belongs in the key; dropping any silently loses rows.
KEYS = ["usage_date", "resource_id", "environment_id", "feature", "channel", "llm_model"]

consumption = (spark.createDataFrame(rows, schema=CONSUMPTION_SCHEMA)
               if rows else spark.createDataFrame([], schema=CONSUMPTION_SCHEMA))
consumption = consumption.withColumn("_loaded_at", F.current_timestamp())

restate_from = END - timedelta(days=RESTATE_DAYS - 1)

if not spark.catalog.tableExists(TBL_CONSUMPTION):
    consumption.write.format("delta").saveAsTable(TBL_CONSUMPTION)
    print(f"{TBL_CONSUMPTION}: created with {consumption.count():,} rows")
else:
    target = DeltaTable.forName(spark, TBL_CONSUMPTION)

    # Replace the restatement window wholesale.
    target.delete(F.col("usage_date") >= F.lit(restate_from))
    (consumption.filter(F.col("usage_date") >= F.lit(restate_from))
     .write.format("delta").mode("append").saveAsTable(TBL_CONSUMPTION))

    # Merge everything older, so history outlives the API's retention.
    older = consumption.filter(F.col("usage_date") < F.lit(restate_from))
    if older.head(1):
        cond = " AND ".join(f"t.{k} <=> s.{k}" for k in KEYS)
        (DeltaTable.forName(spark, TBL_CONSUMPTION).alias("t")
         .merge(older.alias("s"), cond)
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    print(f"{TBL_CONSUMPTION}: restated from {restate_from}, merged {older.count():,} older rows")

capacity = (spark.createDataFrame([capacity_row])
            .withColumn("collected_at", F.current_timestamp())
            .withColumn("collected_date", F.current_date()))
if spark.catalog.tableExists(TBL_CAPACITY):
    (DeltaTable.forName(spark, TBL_CAPACITY).alias("t")
     .merge(capacity.alias("s"),
            "t.collected_date <=> s.collected_date AND t.entitlement_id <=> s.entitlement_id")
     .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
else:
    capacity.write.format("delta").saveAsTable(TBL_CAPACITY)
print(f"{TBL_CAPACITY}: one snapshot for {date.today()}")

if env_rows:
    environments = (spark.createDataFrame(env_rows)
                    .withColumn("_loaded_at", F.current_timestamp()))
    environments.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(TBL_ENVIRONMENT)
    print(f"{TBL_ENVIRONMENT}: {len(env_rows):,} rows (full replace)")

## 5. Check

The tenant capacity figure is the source of truth for the headline. Per-resource consumption is
an attribution of it and will not necessarily add up: consumption that carries no resource is
not returned here. A gap is expected — investigate a large one, do not assume a collection bug.

In [ ]:
for t in (TBL_CONSUMPTION, TBL_CAPACITY, TBL_ENVIRONMENT):
    if spark.catalog.tableExists(t):
        print(f"{t:28s} {spark.table(t).count():>8,} rows")
    else:
        print(f"{t:28s}        - not created")

if spark.catalog.tableExists(TBL_CONSUMPTION):
    spark.sql(f"""
        SELECT  MIN(usage_date) AS earliest, MAX(usage_date) AS latest,
                COUNT(DISTINCT usage_date)  AS days,
                COUNT(DISTINCT resource_id) AS agents,
                ROUND(SUM(billed_credit))     AS billed,
                ROUND(SUM(non_billed_credit)) AS non_billed
        FROM    {TBL_CONSUMPTION}
    """).show(truncate=False)

    print("Inferred harness split - confirm against the Harness column in Power Platform "
          "inventory before reporting it as fact:")
    spark.sql(f"""
        SELECT  COALESCE(harness_hint, 'unknown') AS harness_hint,
                COUNT(*)                      AS rows,
                COUNT(DISTINCT resource_id)   AS agents,
                ROUND(SUM(billed_credit))     AS billed,
                ROUND(SUM(non_billed_credit)) AS non_billed
        FROM    {TBL_CONSUMPTION}
        GROUP BY 1 ORDER BY billed DESC NULLS LAST
    """).show(truncate=False)

    gap = spark.sql(f"SELECT SUM(billed_credit) AS c FROM {TBL_CONSUMPTION}").collect()[0]["c"]
    if capacity_row["consumed"] and gap:
        share = gap / capacity_row["consumed"]
        print(f"per-resource billed total is {share:.0%} of the tenant consumed figure "
              f"({gap:,.0f} of {capacity_row['consumed']:,.0f})")
        print("  note: the windows differ - tenant consumed is life-to-date for the "
              "entitlement, this collection covers DAYS days.")